In [ ]:
import cv2
import pandas as pd
import numpy as np
from pathlib import Path

import sys
import importlib
from pathlib import Path

importlib.invalidate_caches()

ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.features.glcm import build_feature_dataset
from src.models.baseline import train_and_evaluate_rf

print(" Modüller başarıyla yüklendi!")

In [ ]:
ROOT_DIR = Path.cwd().parent
p = ROOT_DIR / "data" / "raw" / "NEU-DET"

In [ ]:
all_images = list(p.rglob('*.jpg'))

In [ ]:
data_list = []

In [ ]:
first = all_images[0]

In [ ]:
for image_path in all_images:
    row = {
        'file_path': str(image_path),
        'defect_class': image_path.parts[-2],
        'split': 'train' if 'train' in image_path.parts else 'validation'
    }
    data_list.append(row)
    img = cv2.imread(str(image_path))
    if img is not None:
        row.update({
            'height': img.shape[0],
            'width': img.shape[1],
            'channels': img.shape[2],
            'pixel_mean': np.mean(img),
            'pixel_std': np.std(img)
        })
    

In [ ]:
df = pd.DataFrame(data_list)

In [ ]:
df

In [ ]:
print("Toplam Satır ve Sütun Sayısı:", df.shape)
print("\nEksik (Bozuk) Değer Sayısı:\n", df.isnull().sum())

# 2. Sınıfların Train/Validation dağılım tablosu
print("\nSınıf Dağılım Matrisi:")
print(pd.crosstab(df['split'], df['defect_class']))

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
defect_classes = df['defect_class'].unique()

fig, axes = plt.subplots(nrows=6, ncols=3, figsize=(9, 14))

for row_idx, cls_name in enumerate(defect_classes):
    sample_paths = df[df['defect_class'] == cls_name]['file_path'].head(3).values

    for col_idx, img_path in enumerate(sample_paths):
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        ax = axes[row_idx, col_idx]
        ax.imshow(image, cmap='gray')
        ax.axis('off')
        if col_idx == 0:
            ax.set_title(cls_name, fontsize=12, fontweight='bold', loc='left')
plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns

plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x='pixel_mean', 
    y='pixel_std',
    hue='defect_class',
    palette='tab10',
    alpha=0.8
)

plt.title('NEU-CLS Kusur Sınıfları: Parlaklık vs. Kontrast Ayrışımı', fontsize=13, fontweight='bold')
plt.xlabel('Ortalama Piksel Değeri (Mean Brightness)', fontsize=11)
plt.ylabel('Piksel Standart Sapması (Contrast)', fontsize=11)

plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)

plt.tight_layout()
plt.show()

In [ ]:
from skimage.feature import graycomatrix, graycoprops

In [ ]:
scratch = df[df['defect_class'] == 'scratches']['file_path'].iloc[0]
crazing = df[df['defect_class'] == 'crazing']['file_path'].iloc[0]

scratch_img = cv2.imread(scratch, cv2.IMREAD_GRAYSCALE)
crazing_img = cv2.imread(crazing, cv2.IMREAD_GRAYSCALE)

glcm_scratch = graycomatrix(scratch_img, distances=[1], angles=[0, np.pi/2], levels=256, symmetric=True, normed=True)
glcm_crazing = graycomatrix(crazing_img, distances=[1], angles=[0, np.pi/2], levels=256, symmetric=True, normed=True)

s_corr_0 = graycoprops(glcm_scratch, 'correlation')[0, 0]
s_corr_90 = graycoprops(glcm_scratch, 'correlation')[0, 1]

c_corr_0 = graycoprops(glcm_crazing, 'correlation')[0, 0]
c_corr_90 = graycoprops(glcm_crazing, 'correlation')[0, 1]

In [ ]:
print(s_corr_0, s_corr_90)


In [ ]:
print(c_corr_0, c_corr_90)

In [ ]:
from tqdm import tqdm

def extract_image_features(image_path):
    """
    Tek bir görseli diskten okur; 1. derece istatistikleri ve 
    çok ölçekli GLCM Haralick özniteliklerini (mean & range) çıkarır.
    """
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
  
    features = {
        'pixel_mean': float(np.mean(img)),
        'pixel_std': float(np.std(img))
    }
    
    distances = [1, 3, 5]
    angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]
    properties = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation']
 
    glcm = graycomatrix(
        img, 
        distances=distances, 
        angles=angles, 
        levels=256, 
        symmetric=True, 
        normed=True
    )

    for prop in properties:
        prop_matrix = graycoprops(glcm, prop)
        
        for d_idx, dist in enumerate(distances):
            angle_values = prop_matrix[d_idx, :]
            
            features[f'{prop}_mean_d{dist}'] = float(np.mean(angle_values))
            
            features[f'{prop}_range_d{dist}'] = float(np.ptp(angle_values))
            
    return features

In [ ]:
feature_rows = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Öznitelikler Çıkarılıyor"):
    img_features = extract_image_features(row['file_path'])
    
    if img_features is not None:
        img_features['file_path'] = row['file_path']
        img_features['defect_class'] = row['defect_class']
        img_features['split'] = row['split']
        feature_rows.append(img_features)

features_df = pd.DataFrame(feature_rows)

In [ ]:
print("Oluşturulan Tablo Boyutu:", features_df.shape)

inspection = features_df.groupby('defect_class')[['correlation_mean_d1', 'correlation_range_d1']].mean()
print("\nKusur Sınıflarına Göre Korelasyon ve Yönlülük Özeti:\n", inspection)

In [ ]:
features_df

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

feature_cols = [c for c in features_df.columns if c not in ['file_path', 'defect_class', 'split']]

train_mask = features_df['split'] == 'train'
val_mask = features_df['split'] == 'validation'

X_train = features_df.loc[train_mask, feature_cols]
y_train = features_df.loc[train_mask, 'defect_class']

X_val = features_df.loc[val_mask, feature_cols]
y_val = features_df.loc[val_mask, 'defect_class']

print(f"Eğitim Seti: {X_train.shape[0]} görsel, {X_train.shape[1]} öznitelik")
print(f"Doğrulama Seti: {X_val.shape[0]} görsel")

rf_model = RandomForestClassifier(
    n_estimators=150,       
    max_depth=12,          
    random_state=42,
    n_jobs=-1            
)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_val)

val_acc = accuracy_score(y_val, y_pred)
print(f"\n✅ Doğrulama Kümesi Doğruluk Oranı (Accuracy): %{val_acc * 100:.2f}\n")
print("--- Sınıflandırma Raporu (Precision / Recall / F1-Score) ---")
print(classification_report(y_val, y_pred))

In [ ]:

plt.figure(figsize=(8, 6))
labels = sorted(y_val.unique())
cm = confusion_matrix(y_val, y_pred, labels=labels)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=labels, yticklabels=labels)
plt.title(f'Baseline Random Forest Confusion Matrix (Acc: %{val_acc*100:.1f})', fontsize=12, fontweight='bold')
plt.xlabel('Modelin Tahmini (Predicted)', fontsize=11)
plt.ylabel('Gerçek Sınıf (Actual)', fontsize=11)
plt.tight_layout()
plt.show()

importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
importances.head(10).plot(kind='barh', color='#2b5c8f')
plt.title('En Belirleyici İlk 10 Doku Özniteliği (Feature Importances)', fontsize=12, fontweight='bold')
plt.xlabel('Önem Skoru (Importance Score)', fontsize=11)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:

features_df = build_feature_dataset(df, save_path="../data/processed/glcm_features.parquet")

results = train_and_evaluate_rf(features_df, model_save_path="../models/rf_baseline.joblib")

print(f"\n Pipeline Tamamlandı! Doğrulama Başarısı: %{results['accuracy'] * 100:.2f}")